# NLP Application: Sentiment Analysis
**COP4023 – Programming Languages**
**Student:** Lovinson Dieujuste

Sentiment analysis of Amazon Alexa customer reviews (`customer_reviewers.tsv`) using an LSTM neural network.

## 0. Setup – install & import packages

In [ ]:
!pip install -q wordcloud nltk textblob

In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import nltk
from nltk.corpus import stopwords
from wordcloud import WordCloud

import sklearn
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

import tensorflow as tf
import keras
# Tokenizer from the keras.preprocessing.text package (exposed via tf.keras in Keras 3)
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from keras.models import Sequential
from keras.layers import Input, Embedding, SpatialDropout1D, LSTM, Dense
from keras.utils import to_categorical

nltk.download('stopwords', quiet=True)
print('pandas', pd.__version__, '| numpy', np.__version__, '| nltk', nltk.__version__,
      '| sklearn', sklearn.__version__, '| tensorflow', tf.__version__, '| keras', keras.__version__)

### requirements.txt
Writes the exact package versions used in this runtime to `requirements.txt`.

In [ ]:
import importlib.metadata as md_
pkgs = ['numpy', 'pandas', 'matplotlib', 'nltk', 'textblob', 'wordcloud',
        'scikit-learn', 'tensorflow', 'keras']
with open('requirements.txt', 'w') as f:
    for p in pkgs:
        try:
            f.write(f'{p}=={md_.version(p)}\n')
        except md_.PackageNotFoundError:
            pass
print(open('requirements.txt').read())

## 1. Data Wrangling

### 1.1 Read the dataset into a DataFrame

In [ ]:
DATA_FILE = 'customer_reviewers.tsv'

# In Colab, upload the file if it is not already in the working directory
if not os.path.exists(DATA_FILE):
    try:
        from google.colab import files
        uploaded = files.upload()          # choose customer_reviewers.tsv
        DATA_FILE = next(iter(uploaded))
    except ImportError:
        raise FileNotFoundError('Place customer_reviewers.tsv next to this notebook.')

df = pd.read_csv(DATA_FILE, sep='\t')
print(df.shape)

### 1.2 First ten rows

In [ ]:
df.head(10)

### 1.3 Verify there are no missing values

In [ ]:
print('Missing values per column (before cleaning):')
print(df.isnull().sum())

# Drop any rows with missing values so the data is complete
df = df.dropna().reset_index(drop=True)

print('\nMissing values per column (after cleaning):')
print(df.isnull().sum())
assert df.isnull().sum().sum() == 0, 'Data still contains missing values'
print('\nNo missing values. Rows:', len(df))

### 1.4 Remove stop words

In [ ]:
try:
    STOP_WORDS = set(stopwords.words('english'))          # NLTK English stop-word list
except LookupError:
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
    STOP_WORDS = set(ENGLISH_STOP_WORDS)                 # fallback if NLTK data is offline
print(len(STOP_WORDS), 'stop words')

def remove_stopwords(data, column='verified_reviews'):
    """Lower-case the reviews, strip punctuation/digits and remove English stop words.
    Returns a DataFrame whose review column contains only non-stop words."""
    def clean(text):
        text = re.sub(r'[^a-zA-Z\s]', ' ', str(text).lower())
        return ' '.join(w for w in text.split() if w not in STOP_WORDS)
    cleaned = data.copy()
    cleaned[column] = cleaned[column].apply(clean)
    return cleaned

df_clean = remove_stopwords(df)
df_clean[['verified_reviews', 'feedback']].head(10)

### 1.5 Word cloud of the most frequent words

In [ ]:
all_words = ' '.join(df_clean['verified_reviews'])
wordcloud = WordCloud(width=1000, height=500, background_color='white',
                      max_words=150, collocations=False).generate(all_words)

plt.figure(figsize=(14, 7))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Most frequent words in customer reviews', fontsize=16)
plt.show()

## 2. Data Engineering

### 2.1 Encode target labels (0 … n_classes-1) with LabelEncoder

In [ ]:
label_encoder = LabelEncoder()
df_clean['label'] = label_encoder.fit_transform(df_clean['feedback'])
print('Classes:', label_encoder.classes_)
print(df_clean['label'].value_counts())

### 2.2 Convert reviews into vector arrays with Tokenizer

In [ ]:
MAX_WORDS = 500   # vocabulary size (matches the Embedding input_dim)

tokenizer = Tokenizer(num_words=MAX_WORDS, split=' ')
tokenizer.fit_on_texts(df_clean['verified_reviews'].values)
X = tokenizer.texts_to_sequences(df_clean['verified_reviews'].values)
X = pad_sequences(X)
print('X shape:', X.shape)
X[:5]

## 3. Model Design – LSTM

In [ ]:
model = Sequential()
model.add(Input(shape=(X.shape[1],)))
try:
    model.add(Embedding(500, 120, input_length=X.shape[1]))
except (TypeError, ValueError):
    # Keras 3 infers input_length from the Input layer
    model.add(Embedding(500, 120))
model.add(SpatialDropout1D(0.4))
model.add(LSTM(176, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(2, activation='softmax'))
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
model.summary()

In [ ]:
total = model.count_params()
trainable = int(sum(np.prod(w.shape) for w in model.trainable_weights))
non_trainable = int(sum(np.prod(w.shape) for w in model.non_trainable_weights))
print(f'Total params: {total:,}')
print(f'Trainable params: {trainable:,}')
print(f'Non-trainable params: {non_trainable:,}')
assert total == 269442 and trainable == 269442 and non_trainable == 0

## 4. Data Splitting (Training / Testing)

### 4.1 One-hot encode labels

In [ ]:
y = to_categorical(df_clean['label'].values, num_classes=2)
print('y shape:', y.shape)
y[:5]

### 4.2 80 / 20 train-test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print('Train:', X_train.shape, y_train.shape)
print('Test :', X_test.shape, y_test.shape)

## 5. Training the Model

In [ ]:
history = model.fit(X_train, y_train, epochs=5, batch_size=32, verbose='auto')

## 6. Evaluate the Model

In [ ]:
y_pred_prob = model.predict(X_test)
y_pred = np.argmax(y_pred_prob, axis=1)
y_true = np.argmax(y_test, axis=1)
print('Predicted:', y_pred[:20])
print('Actual   :', y_true[:20])

## 7. Performed Evaluation – classification report

In [ ]:
print(classification_report(y_true, y_pred, labels=[0, 1],
                            target_names=['Negative (0)', 'Positive (1)'], zero_division=0))

## 8. Client Program
Type a product review to get its predicted sentiment. Type `quit` (or press Enter on an empty line) to stop.

In [ ]:
SENTIMENT = {0: 'Negative', 1: 'Positive'}

def predict_sentiment(review):
    # 1. clean the review from stop words using the implemented function
    cleaned = remove_stopwords(pd.DataFrame({'verified_reviews': [review]}))
    # 2. turn the review's words into a vector
    seq = tokenizer.texts_to_sequences(cleaned['verified_reviews'].values)
    seq = pad_sequences(seq, maxlen=X.shape[1])
    # 3. predict with the trained model
    probs = model.predict(seq, verbose=0)[0]
    label = int(np.argmax(probs))
    return SENTIMENT[label], float(probs[label])

def client_program():
    print('=== Product Review Sentiment Analyzer ===')
    while True:
        review = input('\nEnter your review for the product (or "quit"): ').strip()
        if review == '' or review.lower() == 'quit':
            print('Goodbye!')
            break
        sentiment, confidence = predict_sentiment(review)
        print(f'Predicted sentiment: {sentiment} (confidence {confidence:.2%})')

client_program()